# Standardized detector: Qwen-trained, tested on Qwen and Gemma-2-9B-it

1. **Train** a Qwen2.5-7B detector exactly like `sala-g14-logsumexp-4x128` (six sources, TriviaQA held out, log-sum-exp pooling to
   15 layers, no PCA, 4 x 128 MLP, 40 epochs) **plus standardization**: every one of the 15 x 3584 MLP inputs is z-scored with the
   mean / std of Qwen's source-train rows. The statistics are saved in `model.pt`.
2. **Gemma** answers the first `N` TriviaQA questions; its last-answer-token hidden states `(N, 43, 3584)` are pooled to 15 layers.
3. Gemma is scored twice: with **Qwen's statistics** (what the detector learned) and with **Gemma's own statistics** computed from
   the same `N` answers (`--standardize-on-input`, puts Gemma on the same scale; with N = 10 the statistics are noisy).
4. Control: Qwen's own saved states for the same questions, and a stage-by-stage diagnostic.

Standardization fixes the scale difference between the models. It does not make hidden unit *i* of Gemma mean what unit *i* of
Qwen means, so weak Gemma scores are still possible.

**Inputs:** the features dataset `sala_qwen25_g14_6src_triviaqa_v1` and Gemma (Models -> google / gemma-2 / Transformers /
gemma-2-9b-it). GPU T4 x2, internet on.

In [ ]:
# Clones the repo. Public repo: no credentials needed. If the anonymous clone fails (for example the repo is private
# again) it retries with the GITHUB_TOKEN Kaggle secret, which is passed to git only through an environment header,
# never in the command line or URL, so it cannot show up in tracebacks, logs or the saved clone.
import base64, os, subprocess, urllib.request

REPO, REF = "brainardphilemon/hallucination-probes", "main"   # pin a commit hash for reproducible runs
DEST = "/kaggle/working/hallucination-probes"
URL = f"https://github.com/{REPO}.git"
subprocess.run(["rm", "-rf", DEST], check=True)
clone = subprocess.run(["git", "clone", "-q", URL, DEST], env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
                       capture_output=True, text=True)
if clone.returncode:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Anonymous git clone failed and no GITHUB_TOKEN secret is attached "
                           "(Add-ons -> Secrets). git said: " + clone.stderr.strip()[-300:]) from None
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
           "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader", "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"}
    subprocess.run(["rm", "-rf", DEST], check=True)
    clone = subprocess.run(["git", "clone", "-q", URL, DEST], env=env, capture_output=True, text=True)
    if clone.returncode:
        try:  # 401 = token invalid/expired; 404 = token cannot see this repo (wrong repository or missing Contents: read)
            request = urllib.request.Request(f"https://api.github.com/repos/{REPO}", headers={"Authorization": f"Bearer {token}"})
            status = urllib.request.urlopen(request).status
        except Exception as exc:
            status = getattr(exc, "code", type(exc).__name__)
        raise RuntimeError(f"git clone failed; GitHub API answered {status} for {REPO} with your token. 401: token invalid or "
                           "expired. 404: the token has no read access to this repository (create it with Resource owner "
                           "brainardphilemon, repository hallucination-probes, Contents: Read-only).") from None
    del token, auth, env
subprocess.run(["git", "-C", DEST, "checkout", "-q", REF], check=True)
print(subprocess.run(["git", "-C", DEST, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r /kaggle/working/hallucination-probes/requirements-kaggle.txt
!pip install -q --no-deps -e /kaggle/working/hallucination-probes

In [ ]:
# ---- config -------------------------------------------------------------------------------
N = 10                                                     # Gemma questions (first N of the held-out TriviaQA set)
FEATURES_NAME = "sala_qwen25_g14_6src_triviaqa_v1"
SOURCES = ["tqa", "nq_open", "sciq", "popqa", "web_questions", "hotpotqa"]

In [ ]:
import json
from pathlib import Path

def find_one(name, test, attach=""):
    root = Path("/kaggle/input")
    hits = [p for p in root.rglob(name) if test(p)]
    if len(hits) != 1:
        attached = sorted({str(p.relative_to(root)) for pattern in ("*", "*/*", "*/*/*") for p in root.glob(pattern)})
        raise RuntimeError(f"Expected exactly one {name} under /kaggle/input, found {hits}. {attach}\n"
                           "Attached right now:\n  " + "\n  ".join(attached[:60] or ["(nothing)"]))
    return hits[0]

def is_gemma_9b(config_path):
    c = json.loads(config_path.read_text())
    return c.get("model_type") == "gemma2" and c.get("hidden_size") == 3584 and c.get("num_hidden_layers") == 42

FEATURES = find_one(FEATURES_NAME, lambda p: (p / "generation").is_dir(),
                    "Add Input -> Datasets -> sala-g14-6-source-triviaqa-version-6-artifacts (nvthaai).")
GEMMA = find_one("config.json", is_gemma_9b,
                 "Add Input -> Models -> google / gemma-2 / Transformers / gemma-2-9b-it.").parent
print(FEATURES, GEMMA, sep="\n")

In [ ]:
# 1. Train the standardized Qwen detector (same setup as sala-g14-logsumexp-4x128 + --standardize).
!hprobes train --method mlp --artifact-dir "{FEATURES}" --output-dir /kaggle/working/qwen_std --sources {" ".join(SOURCES)} --targets triviaqa --pooling logsumexp --pool-layers 15 --standardize --hidden-dim 128 --hidden-layers 4 --epochs 40 --seed 42 --device cuda:0

In [ ]:
# 2. First N TriviaQA questions (held out from training) and Gemma's answers + hidden states.
rows = [json.loads(l) for l in (FEATURES / "generation" / "triviaqa" / "rows.jsonl").read_text().splitlines()[:N]]
with open("/kaggle/working/questions.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps({"id": r["id"], "input_text": r["input_text"], "ground_truth_answers": r["ground_truth_answers"]}) + "\n")

In [ ]:
!hprobes generate --model "{GEMMA}" --questions /kaggle/working/questions.jsonl --limit {N} --output-dir /kaggle/working/gemma

In [ ]:
# 3. Gemma scored with Qwen's statistics, then with Gemma's own statistics.
!hprobes predict --model /kaggle/working/qwen_std/model.pt --states /kaggle/working/gemma/states.npy --records /kaggle/working/gemma/records.jsonl --bleurt --device cuda --output /kaggle/working/gemma_qwen_stats.csv
!hprobes predict --model /kaggle/working/qwen_std/model.pt --states /kaggle/working/gemma/states.npy --records /kaggle/working/gemma/records.jsonl --bleurt --standardize-on-input --device cuda --output /kaggle/working/gemma_own_stats.csv

In [ ]:
# 4. Control: Qwen's own saved states and answers for the same questions.
import numpy as np
np.save("/kaggle/working/qwen_states.npy", np.load(FEATURES / "generation" / "triviaqa" / "features.npy", mmap_mode="r")[:N])
with open("/kaggle/working/qwen_records.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps({"id": r["id"], "input": r["input_text"], "generated_output": r["output_text"], "actual_output": r["ground_truth_answers"]}) + "\n")

In [ ]:
!hprobes predict --model /kaggle/working/qwen_std/model.pt --states /kaggle/working/qwen_states.npy --records /kaggle/working/qwen_records.jsonl --bleurt --device cuda --output /kaggle/working/qwen_control.csv

In [ ]:
# Stage-by-stage diagnostic: MLP inputs (after standardization) vs 500 other Qwen TriviaQA states.
import sys
sys.path.insert(0, "/kaggle/working/hallucination-probes/src")   # package was installed after the kernel started
from hprobes.diagnose import report
feats = np.load(FEATURES / "generation" / "triviaqa" / "features.npy", mmap_mode="r")
gemma = np.load("/kaggle/working/gemma/states.npy")
_ = report("/kaggle/working/qwen_std/model.pt",
           {"qwen_reference_500": feats[N:N + 500], "qwen_first_N": feats[:N],
            "gemma_qwen_stats": gemma, "gemma_own_stats": gemma},
           reference="qwen_reference_500", device="cuda", standardize_on_input=["gemma_own_stats"])
print("\nToken each Gemma hidden state was taken from:")
for line in open("/kaggle/working/gemma/records.jsonl"):
    r = json.loads(line)
    print(f"  {r['generated_output']!r:28} last token {r.get('last_token')!r}")

In [ ]:
import pandas as pd
summary = json.loads(Path("/kaggle/working/qwen_std/summary.json").read_text())
print("Standardized Qwen detector: TriviaQA AUROC", round(summary["target_metrics"]["triviaqa"]["auroc"], 4),
      "| source-test AUROC", round(summary["source_test_metrics"]["auroc"], 4), "| threshold", round(summary["threshold"], 4))
show = ["generated_output", "bleurt_score", "bleurt_label", "probability_correct", "verdict"]
for name in ("qwen_control", "gemma_qwen_stats", "gemma_own_stats"):
    t = pd.read_csv(f"/kaggle/working/{name}.csv")
    print(f"\n{name}: detector agrees with BLEURT on {int(t.detector_agrees_with_bleurt.sum())}/{len(t)}; "
          f"P(correct) min/median/max = {t.probability_correct.min():.3f}/{t.probability_correct.median():.3f}/{t.probability_correct.max():.3f}")
    display(t[show].round(3))